In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import os

import warnings
warnings.filterwarnings('ignore')

In [2]:
REQUESTS_HEADER = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://www.google.com',
    'DNT': '1', 
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
}

INFOS = []

class FullTextScrapper:
    def __init__(self) -> None:
        pass
    
    def filter_text(self, text):
        return text
    
    def get_full_text_by_url(self, url):  
        #print("Getting full text from: ", url)
        response = requests.get(url, headers=REQUESTS_HEADER, timeout=5)
        soup = BeautifulSoup(response.content, 'html.parser')
        text = ''
        for parag in soup.find_all('p'):
            text += parag.get_text() + '\n'
        return text
    
    def create_df_from_full_text(self, df, ticker:str, temp_df=None, save_count:int=100):
        dates = []
        errors = []
        full_texts = []
        if temp_df is not None:
            errors = temp_df.errors.tolist()
            dates = temp_df.time.tolist()
            full_texts = temp_df.news.tolist()    
            df = df.loc[df.index > temp_df.index[-1]]    
            print("Tamanho do dataframe temporário: ",len(temp_df))
        print("Tamanho do dataframe faltante: ",len(df))
        count = 0
        for news_index in range(len(df)):
            url = df.url[news_index]
            try:
                full_text = self.get_full_text_by_url(url)
                errors.append(None)
            except Exception as e:
                full_text = ''
                errors.append(e)
            full_texts.append(full_text)
            dates.append(df.index[news_index])
            count +=1
            if count >= save_count:
                temp_df = pd.DataFrame({"time":dates, "news":full_texts, "errors":errors})
                temp_df.to_csv(f"temps_texts/{ticker}.csv")
                print(f"Last updated for {ticker}: {datetime.now()}")
                count = 0
        temp_df = pd.DataFrame({"time":dates, "news":full_texts, "errors":errors})
        temp_df.to_csv(f"temps_texts/{ticker}.csv")
        temp_df.index = pd.to_datetime(temp_df.time)
        return temp_df
            

ft = FullTextScrapper()

In [3]:

folder = 'news'
files = os.listdir(folder)
tickers = [f.split("_")[0] for f in files if os.path.isfile(os.path.join(folder, f))]

print(tickers)


['BA', 'MCD', 'JNJ', 'INTC', 'BAC', 'AMZN', 'NVDA', 'ORCL', 'DIS', 'GS', 'GOOG', 'MSFT', 'TSLA', 'META', 'AAPL', 'V', 'PFE', 'KO', 'GE', 'CAT', 'XOM', 'CVX', 'DE', 'F', 'WMT']


In [4]:

for ticker in tickers:
    print(f"\nGetting info for {ticker}")
    original_df_path = f"news/{ticker}_2024-04-01-2024-09-01.csv"
    original_df =  pd.read_csv(original_df_path, index_col=0)
    original_df.index = pd.to_datetime(original_df.index)
    start_date = None
    temp_df = None
    try:
        temp_df = pd.read_csv(f"temps_texts/{ticker}.csv", index_col=0)
        temp_df.index = pd.to_datetime(temp_df.time)
        start_date = temp_df.index[-1]
    except:pass
    print(f"start_date: {start_date}")
    df_full_news = ft.create_df_from_full_text(original_df, ticker, temp_df)
    df_cleaned = df_full_news.loc[~df_full_news.index.duplicated(keep='first')]
    #(pd.DataFrame()).to_csv(f"temps_texts/{ticker}.csv")
    
    df_to_save = pd.concat([original_df, df_cleaned], axis=1)
    df_to_save.to_csv(f"full_texts/{ticker}_full_texts_2024-04-01-2024-09-01.csv")


Getting info for BA
start_date: 2024-08-31 03:46:20
Tamanho do dataframe temporário:  1998
Tamanho do dataframe faltante:  0

Getting info for MCD
start_date: 2024-08-28 13:14:00
Tamanho do dataframe temporário:  79
Tamanho do dataframe faltante:  0

Getting info for JNJ
start_date: 2024-08-30 17:47:55
Tamanho do dataframe temporário:  670
Tamanho do dataframe faltante:  0

Getting info for INTC
start_date: 2024-08-30 19:21:06
Tamanho do dataframe temporário:  1071
Tamanho do dataframe faltante:  0

Getting info for BAC
start_date: 2024-08-31 22:39:00
Tamanho do dataframe temporário:  2568
Tamanho do dataframe faltante:  0

Getting info for AMZN
start_date: 2024-08-31 09:15:00
Tamanho do dataframe temporário:  2059
Tamanho do dataframe faltante:  0

Getting info for NVDA
start_date: 2024-08-31 22:39:00
Tamanho do dataframe temporário:  8784
Tamanho do dataframe faltante:  0

Getting info for ORCL
start_date: 2024-08-30 20:59:00
Tamanho do dataframe temporário:  467
Tamanho do datafram